# TinyCeNN-LM — Colab Training, Hugging Face Publish & Remote Test

This notebook runs the complete TinyCeNN-LM experiment on a Colab GPU:

1. checks the GPU;
2. clones or updates `vtavakkoli/TinyCeNN-LM`;
3. installs the package into the **same Python interpreter used by the notebook kernel**;
4. explicitly adds `src/` to `sys.path` and verifies `import tinycenn_lm`;
5. logs into Hugging Face securely;
6. trains the CeNN adapter on FineWeb;
7. checks training health;
8. publishes the best checkpoint to Hugging Face;
9. downloads that published checkpoint again;
10. runs small generation and numerical sanity tests.

> **Security:** store your Hugging Face write token in **Colab → Secrets** as `HF_TOKEN`. Never hard-code a real token in a public notebook.


In [ ]:
# 1) Check the runtime and GPU
import sys, subprocess, platform

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    subprocess.run(["nvidia-smi"], check=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    raise RuntimeError(
        "No NVIDIA GPU is visible. In Colab choose Runtime → Change runtime type → GPU."
    )


In [ ]:
# 2) Clone or update the repository
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/vtavakkoli/TinyCeNN-LM.git"
REPO_DIR = Path("/content/TinyCeNN-LM")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print("Working directory:", Path.cwd())


In [ ]:
# 3) Install into THIS notebook kernel and verify the package import
# Using sys.executable avoids the common Jupyter/Colab problem where `!python`
# points at a different interpreter from the active notebook kernel.
import sys, subprocess, importlib
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "huggingface_hub"],
    check=True,
)

# Editable installs should already work, but explicitly adding src/ makes this
# robust in notebook kernels that do not refresh site-packages immediately.
SRC_DIR = REPO_DIR / "src"
src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

importlib.invalidate_caches()
import tinycenn_lm
from tinycenn_lm import build_from_adapter

print("tinycenn_lm import: PASS")
print("Package file:", tinycenn_lm.__file__)


## 4) Login to Hugging Face Hub

Create a Hugging Face **write** token and add it to Colab's Secrets panel using the name `HF_TOKEN`.

If the secret is unavailable, the cell falls back to Hugging Face's interactive login.


In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login, HfApi

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN secret not found; using interactive login.")
    login()

api = HfApi()
identity = api.whoami()
hf_user = identity["name"]
print("Logged in as:", hf_user)


## 5) Training configuration

The default is a **1M-token smoke run**. Once it is healthy, change `MAX_TOKENS` to `10_000_000`, then optionally `50_000_000`.

These settings are conservative for a Colab T4/L4/A100.


In [ ]:
# Training settings
MAX_TOKENS = 1_000_000
CONTEXT_LENGTH = 256
BATCH_SIZE = 8
GRAD_ACCUM = 4
CENN_STEPS = 4
LEARNING_RATE = 0.002

EVAL_EVERY = 25
EVAL_BATCHES = 8
EVAL_BATCH_SIZE = 4

OUTPUT_DIR = REPO_DIR / "checkpoints" / "colab-tinycenn-base"
print("Output directory:", OUTPUT_DIR)


In [ ]:
# 6) Train the CeNN adapter
# Uses the same interpreter as the notebook, so the editable package is guaranteed visible.
import shlex, subprocess, sys

cmd = [
    sys.executable, "scripts/train_adapter.py",
    "--max-tokens", str(MAX_TOKENS),
    "--context-length", str(CONTEXT_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--steps", str(CENN_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--eval-every", str(EVAL_EVERY),
    "--eval-batches", str(EVAL_BATCHES),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--output-dir", str(OUTPUT_DIR),
]

print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
# 7) Inspect training health
import json
from pathlib import Path

report_path = OUTPUT_DIR / "training_report.json"
if not report_path.exists():
    raise FileNotFoundError(f"Training report not found: {report_path}")

report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))

status = report.get("status")
if status == "diverged":
    raise RuntimeError("Training diverged. Do not publish this checkpoint.")

print("\nTraining status:", status)
print("Initial eval loss:", report.get("initial_eval_loss"))
print("Best eval loss:", report.get("best_eval_loss"))
print("Best perplexity:", report.get("best_perplexity"))
print("Relative improvement:", report.get("relative_best_improvement"))


## 8) Prepare the best checkpoint

The trainer saves a `-best` adapter whenever monitoring loss improves. If a separate best directory does not exist, the final stable adapter is used.


In [ ]:
# Prepare the checkpoint folder for Hugging Face
from pathlib import Path
import shutil
from transformers import AutoTokenizer

best_dir = Path(str(OUTPUT_DIR) + "-best")
final_dir = OUTPUT_DIR
publish_dir = best_dir if best_dir.exists() else final_dir

required = [publish_dir / "cenn_adapter.pt", publish_dir / "cenn_config.json"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"Required adapter files missing: {missing}")

# Put tokenizer assets alongside the adapter.
tokenizer = AutoTokenizer.from_pretrained("arnir0/Tiny-LLM", use_fast=True)
tokenizer.save_pretrained(publish_dir)

# Put the full training report beside the selected adapter.
if report_path.resolve() != (publish_dir / "training_report.json").resolve():
    shutil.copy2(report_path, publish_dir / "training_report.json")

model_card = f'''---
base_model: arnir0/Tiny-LLM
library_name: transformers
pipeline_tag: text-generation
tags:
- cenn
- tiny-llm
- language-modeling
- recurrent-neural-network
- parameter-efficient
- research
---

# TinyCeNN-LM Base

CeNN residual adapter trained on top of `arnir0/Tiny-LLM`.

- CeNN recurrent steps: {CENN_STEPS}
- Context length: {CONTEXT_LENGTH}
- Training token budget: {MAX_TOKENS:,}
- Health status: {report.get("status")}
- Initial eval loss: {report.get("initial_eval_loss")}
- Best eval loss: {report.get("best_eval_loss")}
- Best perplexity: {report.get("best_perplexity")}

This model repository contains TinyCeNN adapter weights/configuration, tokenizer
metadata, and the training report. Reconstruct the model with the TinyCeNN-LM
GitHub code.
'''
(publish_dir / "README.md").write_text(model_card, encoding="utf-8")

print("Publishing from:", publish_dir)
for p in sorted(publish_dir.iterdir()):
    print(" -", p.name)


In [ ]:
# 9) Create/update the Hugging Face model repository and upload the checkpoint
HF_MODEL_NAME = "TinyCeNN-LM-Base"
HF_REPO_ID = f"{hf_user}/{HF_MODEL_NAME}"
HF_PRIVATE = False  # set True for a private model repository

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=HF_PRIVATE,
    exist_ok=True,
)

api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(publish_dir),
    commit_message=f"Upload TinyCeNN-LM checkpoint ({MAX_TOKENS:,} training tokens)",
)

print("Uploaded model:")
print(f"https://huggingface.co/{HF_REPO_ID}")


## 10) Download the uploaded model back from Hugging Face

The following tests intentionally use a fresh Hugging Face snapshot instead of the local training directory. This verifies that the published artifact is self-consistent.


In [ ]:
# Download the published adapter from Hugging Face
from huggingface_hub import snapshot_download

downloaded_adapter = snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print("Downloaded to:", downloaded_adapter)


In [ ]:
# 11) Re-verify the local TinyCeNN package path, then load the published model
# This makes the cell safe even if the notebook kernel changed its working directory.
import sys, importlib
from pathlib import Path

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()

import tinycenn_lm
from tinycenn_lm import build_from_adapter

print("TinyCeNN package:", tinycenn_lm.__file__)


In [ ]:
# 12) Small deterministic generation tests
import torch
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type == "cuda" else torch.float32)
)

model = build_from_adapter(downloaded_adapter, device=device, dtype=dtype)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(downloaded_adapter)

prompts = [
    "The capital of Austria is",
    "Artificial intelligence can help",
    "A small language model",
    "In the future, efficient AI",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print("\nPROMPT:", prompt)
    print(text)


In [ ]:
# 13) Numerical sanity test on the downloaded Hugging Face model
import math

test_text = "TinyCeNN-LM is a compact research language model."
batch = tokenizer(test_text, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model(
        **batch,
        labels=batch["input_ids"],
        use_cache=False,
    )

loss = float(outputs.loss.detach().cpu())
ppl = math.exp(min(loss, 20.0))

print(f"Sanity loss: {loss:.4f}")
print(f"Sanity perplexity: {ppl:.2f}")

assert math.isfinite(loss), "Non-finite loss after reloading from Hugging Face."
print("Reload + inference sanity check: PASS")


## Next run

If the 1M-token run is healthy:

- change `MAX_TOKENS = 10_000_000`;
- rerun training through upload;
- only then consider 50M tokens.

The uploaded model can also be tested from Docker:

```bash
HF_MODEL_REPO=<your-hf-user>/TinyCeNN-LM-Base docker compose run --rm test-hf
```
